# 03 · Evaluasi dan analisis hasil (T4-A, Raabin-WBC)

Notebook ini **hanya membaca** hasil run dari Drive (`logs/` dan `results/`). Tidak melatih apa pun, tidak butuh GPU (Runtime boleh CPU), dan tidak menulis ke log.

**Cara pakai:** buka lewat *File → Save a copy in Drive* supaya tiap orang punya salinan sendiri. Jalankan sel 1 dulu, lalu sel milikmu.

| Sel | Isi | Dijalankan oleh |
|---|---|---|
| 1 | Muat log dan cek kelengkapan seed | Semua |
| 2 | Tabel hasil utama (rerata ± SD, 3 seed) | Ketua |
| 3 | Dice per jenis sel di set uji | Anggota 4 |
| 4 | Efisiensi (parameter, latensi) | Anggota 4 |
| 5 | Uji berpasangan U-Net vs DeepLabV3+ | Ketua |
| 6 | Tabel ablasi (validasi, 1 seed) | Ketua |

Belum termasuk di notebook ini: galeri kesalahan, Grad-CAM, dan uji eksternal.

In [2]:
# ===== SEL 1: setup dan muat log (semua anggota) =====
from google.colab import drive; drive.mount('/content/drive')
import glob, pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"
import os; os.makedirs(f"{DRIVE}/results", exist_ok=True)

files = glob.glob(f"{DRIVE}/logs/log_*.csv")
log = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
log = log.drop_duplicates(subset="run_id")                       # salinan log bernomor tidak terhitung ganda
log = log[(log.ablation != "smoke_test") & (log.status == "ok")]
m = log[log.ablation == "main"].copy()

print("Run ok (tanpa smoke_test):", len(log), "| run main:", len(m))
print(m.groupby("arch").seed.apply(lambda s: sorted(s)))
assert all(sorted(g.seed) == [42, 43, 44] for _, g in m.groupby("arch")), \
    "Seed main tidak lengkap atau ada run ganda: cek tabel di atas"
assert m.arch.nunique() == 2, "Harus ada dua arsitektur di run main"


Mounted at /content/drive
Run ok (tanpa smoke_test): 17 | run main: 6
arch
deeplabv3plus    [42, 43, 44]
unet             [42, 43, 44]
Name: seed, dtype: object


In [3]:
# ===== SEL 2: tabel hasil utama, rerata ± SD antar-seed (Ketua) =====
cols = [c for c in m.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
def fmt(g, c): return f"{g[c].mean():.4f} ± {g[c].std(ddof=1):.4f}"
tab = pd.DataFrame({arch: {c: fmt(g, c) for c in cols} for arch, g in m.groupby("arch")})
print(tab.to_string())
tab.to_csv(f"{DRIVE}/results/tabel_hasil_utama.csv")

# epoch berhenti tiap seed (untuk catatan keterbatasan early stopping)
print()
print(m.sort_values(["arch", "seed"])[["owner", "arch", "seed", "best_epoch", "epochs_run"]].to_string(index=False))


                  deeplabv3plus             unet
val_dice_bg     0.9942 ± 0.0002  0.9942 ± 0.0003
val_iou_bg      0.9886 ± 0.0003  0.9886 ± 0.0006
val_dice_cyto   0.8929 ± 0.0021  0.8923 ± 0.0029
val_iou_cyto    0.8152 ± 0.0030  0.8151 ± 0.0034
val_dice_nuc    0.9716 ± 0.0007  0.9722 ± 0.0005
val_iou_nuc     0.9450 ± 0.0012  0.9461 ± 0.0011
val_miou        0.9162 ± 0.0015  0.9166 ± 0.0012
val_hd95_cyto   9.3215 ± 0.2230  8.8244 ± 0.4886
val_hd95_nuc    4.2878 ± 0.0949  4.0308 ± 0.1053
test_dice_bg    0.9939 ± 0.0000  0.9939 ± 0.0002
test_iou_bg     0.9880 ± 0.0000  0.9880 ± 0.0003
test_dice_cyto  0.9088 ± 0.0010  0.9089 ± 0.0030
test_iou_cyto   0.8380 ± 0.0016  0.8382 ± 0.0040
test_dice_nuc   0.9699 ± 0.0002  0.9696 ± 0.0008
test_iou_nuc    0.9420 ± 0.0003  0.9414 ± 0.0014
test_miou       0.9226 ± 0.0006  0.9225 ± 0.0016
test_hd95_cyto  8.6835 ± 0.2749  9.7969 ± 1.8126
test_hd95_nuc   5.6417 ± 0.5587  5.3825 ± 1.3682

owner          arch  seed  best_epoch  epochs_run
Iklil deeplabv3plu

In [10]:
# ===== SEL 3: Dice per jenis sel di set uji (Anggota 4; Ketua juga jalankan sebelum Sel 5) =====
import os
paths = {os.path.basename(p).replace("_test_per_image.csv", ""): p
         for p in glob.glob(f"{DRIVE}/**/*_test_per_image.csv", recursive=True)}
hilang = [r for r in m.run_id if r not in paths]
print("Berkas hasil ditemukan:", len(paths), "| run main:", len(m), "| hilang:", hilang)
assert not hilang, "Ada berkas hasil yang hilang (lihat daftar di atas)"

frames = []
for r in m.itertuples():
    d = pd.read_csv(paths[r.run_id])
    d["arch"], d["seed"] = r.arch, r.seed
    frames.append(d)
res = pd.concat(frames, ignore_index=True)

# rata-rata per (arsitektur, seed, jenis sel), lalu rerata dan SD antar-seed
per_cls = (res.groupby(["arch", "seed", "cls"])[["dice_cyto", "dice_nuc"]].mean()
              .groupby(["arch", "cls"]).agg(["mean", "std"]).round(3))
print(per_cls.to_string())
print("\nJumlah sel uji per jenis:")
print(res[(res.arch == res.arch.iloc[0]) & (res.seed == 42)].groupby("cls").size().to_string())
per_cls.to_csv(f"{DRIVE}/results/dice_per_jenis_sel.csv")

Berkas hasil ditemukan: 6 | run main: 6 | hilang: []
                         dice_cyto        dice_nuc       
                              mean    std     mean    std
arch          cls                                        
deeplabv3plus Eosinophil     0.934  0.004    0.957  0.002
              Lymphocyte     0.853  0.002    0.977  0.001
              Monocyte       0.924  0.000    0.977  0.001
              Neutrophil     0.938  0.002    0.961  0.001
unet          Eosinophil     0.934  0.002    0.960  0.002
              Lymphocyte     0.849  0.012    0.976  0.002
              Monocyte       0.923  0.002    0.977  0.000
              Neutrophil     0.943  0.001    0.960  0.002

Jumlah sel uji per jenis:
cls
Eosinophil    13
Lymphocyte    31
Monocyte      33
Neutrophil    31


In [ ]:
# ===== SEL 4: efisiensi, parameter dan latensi (Anggota 4) =====
eff_cols = ["owner", "arch", "seed", "params_m", "latency_gpu_ms", "latency_cpu_ms"]
eff = m[eff_cols].sort_values(["arch", "seed"])
print(eff.to_string(index=False))
print()
print(eff.groupby("arch")[["params_m", "latency_gpu_ms", "latency_cpu_ms"]].agg(["mean", "std"]).round(2).to_string())
print("\nCatatan: latensi GPU/CPU diukur batch 1 setelah warm-up, rerata 50 citra.")
print("GPU yang dipakai:", sorted(m.gpu.dropna().unique()))
eff.to_csv(f"{DRIVE}/results/efisiensi_main.csv", index=False)


In [11]:
# ===== SEL 5: uji berpasangan per citra, U-Net vs DeepLabV3+ (Ketua) =====
from scipy.stats import wilcoxon
# rata-rata 3 seed per citra untuk tiap arsitektur, lalu bandingkan citra yang sama
avg = res.groupby(["arch", "name"])[["dice_cyto", "dice_nuc"]].mean().unstack(0)
rng = np.random.default_rng(42)
for c in ["dice_cyto", "dice_nuc"]:
    d = (avg[c]["unet"] - avg[c]["deeplabv3plus"]).dropna()
    boot = [rng.choice(d.values, len(d)).mean() for _ in range(5000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    p = wilcoxon(d).pvalue if (d != 0).any() else float("nan")
    print(f"{c}: n={len(d)} | selisih rerata (U-Net - DeepLab) = {d.mean():+.4f} "
          f"| CI95% bootstrap [{lo:+.4f}, {hi:+.4f}] | Wilcoxon p = {p:.4f}")
print("\nCI yang mencakup 0 = tidak ada bukti perbedaan. p kecil dengan selisih ~0,001 tetap bukan perbedaan bermakna.")


dice_cyto: n=108 | selisih rerata (U-Net - DeepLab) = +0.0001 | CI95% bootstrap [-0.0034, +0.0044] | Wilcoxon p = 0.1412
dice_nuc: n=108 | selisih rerata (U-Net - DeepLab) = -0.0003 | CI95% bootstrap [-0.0013, +0.0007] | Wilcoxon p = 0.1514

CI yang mencakup 0 = tidak ada bukti perbedaan. p kecil dengan selisih ~0,001 tetap bukan perbedaan bermakna.


In [5]:
# ===== SEL 6: tabel ablasi, validasi, 1 seed (Ketua) =====
ab = log[log.ablation != "main"].copy()
ab["dihentikan_dini"] = ab.epochs_run < ab.epochs
show = ["ablation", "factor_value", "arch", "best_epoch", "epochs_run", "dihentikan_dini",
        "val_dice_cyto", "val_dice_nuc", "val_miou", "val_hd95_cyto"]
ab = ab.sort_values(["ablation", "factor_value"])[show].round(4)
print(ab.to_string(index=False))
ab.to_csv(f"{DRIVE}/results/tabel_ablasi.csv", index=False)
print("\nPERHATIAN: baris dengan dihentikan_dini=True tidak sebanding dengan baseline (latihan terpotong).")
print("Gunakan ce_p20 dan off_p20 untuk perbandingan loss CE dan warna nonaktif.")
print("Ablasi 'split' membandingkan dua set validasi berbeda (random bocor vs grouped bersih).")


  ablation          factor_value          arch  best_epoch  epochs_run  dihentikan_dini  val_dice_cyto  val_dice_nuc  val_miou  val_hd95_cyto
  baseline deeplabv3plus_ce_dice deeplabv3plus          18          20            False         0.8933        0.9720    0.9166         9.4114
  baseline          unet_ce_dice          unet          17          20            False         0.8963        0.9724    0.9188         8.5563
 color_aug                   off          unet           6          11             True         0.8770        0.9704    0.9071        11.8738
 color_aug               off_p20          unet          17          20            False         0.8963        0.9738    0.9197         9.1328
      loss                    ce          unet           9          14             True         0.8871        0.9704    0.9122        10.4190
      loss                ce_p20          unet          17          20            False         0.8953        0.9719    0.9182         8.7390
      